# MMA 662 - Assignment 2

Instructions
Remark: Assignment 2 has two parts which you need to model and implement in Python
using Gurobi package. The data is provided in a Jupyter Notebook file “Assignment 2-Work
Sheet.ipynb”. Please report your results in your Notebook file with codes in it, and clearly outline
your answers. Lack of clarity may affect your grade and complaints on this matter will not be
entertained.
1 The Apportionment Problem: It’s Political
The problem of fairly allocating parliamentary seats is a political matter in democratic societies1.
Suppose a country with a total population of N = P
i pi tries to allocate its s available seats to
its provinces, each of which has a population of pi. The “fair” way to distribute these seats is to
allocate each province a seat quota proportional to its population:

<p>
qi = s × pi
N = s × pi
P
i pi
.

<p>
Although this solution seems appealing, it ignores the fact that the number of allocated seats
should be an integer. For instance, in a country with 30 million population and 130 parliament
seats, seat quota for a province with a population of 5 million is qi = 5×130
30 = 21.66, which is not
possible. The question then is how to allocate the seats “fairly”. There are two methods to address
this question. The first is a widely-used method that minimizes the sum of the gaps between actual
and ideal allocations. The second one minimizes the maximum gap.
Dataset
The parliament of a fictitious country has 601 seats to be allocated to 20 provinces with different
populations.
The dataset is in Table 1 (it is also available in the Jupyter Notebook for this assignment.) The
quota variable (qi) is rounded to 5 decimals and available in vector “Quota” in the Notebook. (To
make sure everyone is working with the same data, do not calculate the quotas again yourself.)
Also, to prevent indexing issues when coding, we indexed provinces from P 0 to P 19



Installing Gurobi package:

In [1]:
# Uncomment if working with Colab:
# !pip install gurobipy
import gurobipy as gp
from gurobipy import *

import numpy as np
import pandas as pd

## 1: The Apportionment Problem.

In this method, the goal is to minimize the sum of absolute gaps between the number of allocated
seats to each province and its quota while leaving no seats unfilled. That is, we seek to find allocations a0 to a19 to
min
a0,··· ,a19
19X
i=0
|qi − ai| (1)
s.t.
19X
i=0
ai = s.
This is a constrained nonlinear program. But one can “linearize” this program which makes it
possible to use a mixed integer programming algorithm to estimate the optimal variables. To do
so, define zi = |qi − ai| for i = 0, · · · 19. So now we can rewrite the original program in (1), as
min
19X
i=0
zi
s.t. zi = |qi − ai| i = 0, · · · 19
19X
i=0
ai = s.
Although the objective function is now linear, we have nonlinear constraints. To linearize the
constraints, we can substitute zi = |qi − ai| with two constraints: zi ≥ qi − ai AND zi ≥ −(qi − ai).
Recall that ai’s (allocation variables) are integer variables, and zi’s (gap variables) are continuous
variables.
(a) Explain why this linearization is valid. How many decision variables and how many con-
straints does this mixed integer program have?
(b) Find the minimum sum of absolute gaps using the optimal allocation. Your code must be
clear and easy to parse

Dataset:


In [ ]:
# Population list:
Pop = np.array([5300000, 9200000, 5400000, 8000000, 3200000, 
                8800000, 8100000, 10100000, 3600000, 6200000, 
                11700000, 5800000, 7000000, 6300000, 4800000, 
                10700000, 7000000, 9400000, 6400000, 9100000])

# Quota (q_i) list:
Quota = np.array([21.80219, 37.84531, 22.21355, 32.90897, 13.16359, 
                  36.19986, 33.32033, 41.54757, 14.80903, 25.50445, 
                  48.12936, 23.859, 28.79535, 25.91581, 19.74538, 
                  44.01574, 28.79535, 38.66804, 26.32717, 37.43395])

# Province list:
Province = np.array(['P0', 'P1', 'P2', 'P3', 'P4', 
                      'P5', 'P6', 'P7', 'P8', 'P9', 
                      'P10', 'P11', 'P12', 'P13', 'P14', 
                      'P15', 'P16', 'P17', 'P18', 'P19'])

# Total seats:
s = 601

### 1.1: Minimize the sum of absolute gaps.

In [ ]:
# Uncomment if working with Colab:
import gurobipy as gp
from gurobipy import *

import numpy as np
import pandas as pd

# Population list:
Pop = np.array([5300000, 9200000, 5400000, 8000000, 3200000, 
                8800000, 8100000, 10100000, 3600000, 6200000, 
                11700000, 5800000, 7000000, 6300000, 4800000, 
                10700000, 7000000, 9400000, 6400000, 9100000])

# Quota (q_i) list:
Quota = np.array([21.80219, 37.84531, 22.21355, 32.90897, 13.16359, 
                  36.19986, 33.32033, 41.54757, 14.80903, 25.50445, 
                  48.12936, 23.859, 28.79535, 25.91581, 19.74538, 
                  44.01574, 28.79535, 38.66804, 26.32717, 37.43395])

# Province list:
Province = np.array(['P0', 'P1', 'P2', 'P3', 'P4', 
                      'P5', 'P6', 'P7', 'P8', 'P9', 
                      'P10', 'P11', 'P12', 'P13', 'P14', 
                      'P15', 'P16', 'P17', 'P18', 'P19'])

# Total seats:
s = 601

# Your solution for 1.1

#suppress console output
env = gp.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()

#setup problem
prob_1_1=gp.Model("Question 1", env=env)
#prob_1_1=gp.Model("Question 1")

#create variables to define matrix of variables
Qlen=len(Quota)
Plen=len(Province)
Poplen=len(Pop)

########## Create function variables ##########
#Create variables for the the allocation
a=prob_1_1.addVars(Qlen,vtype=GRB.INTEGER, name = ["Allocation "+i for i in Province])
#create variable for the difference
z=prob_1_1.addVars(Qlen,vtype=GRB.CONTINUOUS, name = ["Abs Diff "+i for i in Province])

########## Set objective function ########## 
prob_1_1.setObjective(sum(z[i] for i in range(Qlen)),GRB.MINIMIZE)

########## Set Constraints ##########
## z is  |quota - allocation|

for i in range(Qlen):  
    D=prob_1_1.addVar(lb=-GRB.INFINITY,ub=GRB.INFINITY)
    prob_1_1.addConstr(D==Quota[i]-a[i])
    prob_1_1.addGenConstrAbs(z[i],D,"abscon")
    
## all allocation must be filled
prob_1_1.addConstr(sum(a[i] for i in range(Qlen))==s)

########## Optimize the Problem ##########
prob_1_1.optimize()

########## Print Results ##########
#print objective Coefficiencts
p1_1dict={}
alist=[]
zlist=[]
for i in a:
    alist.append(a[i].X)
    zlist.append(z[i].X)
p1_1dict["allocated"]=alist
p1_1dict["abs difference"]=zlist
df1_1=pd.DataFrame(p1_1dict,index=Province)
df1_1

In [ ]:
print("The optimized absolute difference between the quota and the allocation is {:,}".format(round(prob_1_1.objVal,3)))

## <font color=#FF0000> Questions</font>
### (a) Explain why this linearization is valid. How many decision variables and how many con-straints does this mixed integer program have?

The linearization of this prolem is valid as instead of having ambiguous values for $a$ (the absolute value allows a to be positve or negative) we are replacing the ambiguity with 20 decision variables $z$ (one for each province). $z$ will always be positive or zero regarless of $a$ therefore removing the non-linearity. We retain the information of the absolute difference between $q$ and $a$ by adding constraints. <br>

Since we only care about the positive difference between $q$ and $a$ the constraint linearization is valie. No matter the values of q<sub>i</sub> or a<sub>i</sub>, so long as q<sub>i</sub> != a<sub>i</sub> the resulting z<sub>i</sub> will always be larger some positive number. <br>
if q<sub>i</sub> > a<sub>i</sub>  then  z<sub>i</sub> > x , such that x>0 <br>
if q<sub>i</sub> < a<sub>i</sub>  then  z<sub>i</sub> < -x , such that -x>0 <br>
if q<sub>i</sub> = a<sub>i</sub>  then  z<sub>i</sub> = 0 <br><br>
The number of decsion variables has increased by 1 since b is a binary variablefor the big M method. Therefore in total there are **41 deciscion variables.**<br>
 20 variables for the allocation of representatives ( a<sub>i</sub> i=0,1..19 ) <br>
 20 variables for the difference between the abolute difference of allocation and quota ( z<sub>i</sub> i=0,1..19 ) <br>
 1 dummy variable to "linearize" the absolute difference between allocation and quota (D) <br><br>
 
 **There are a total of 2 constraints:**<br>
 z<sub>i</sub> = |where q<sub>i</sub>-a<sub>i</sub>| i=0,1..19 <br>
 sum (a<sub>i</sub>) = s , where s is the total number of representatives<br>
 
    

### (b) Find the minimum sum of absolute gaps using the optimal allocation. Your code must be clear and easy to parse.

The minimum sum of the abolute gaps is **4.616**

### 1.2: Min-max.

In [ ]:
# Your solution for 1.2

In [ ]:
# Population list:
import gurobipy as gp
from gurobipy import *
import numpy as np
import pandas as pd
Pop = np.array([5300000, 9200000, 5400000, 8000000, 3200000, 
                8800000, 8100000, 10100000, 3600000, 6200000, 
                11700000, 5800000, 7000000, 6300000, 4800000, 
                10700000, 7000000, 9400000, 6400000, 9100000])

# Quota (q_i) list:
Quota = np.array([21.80219, 37.84531, 22.21355, 32.90897, 13.16359, 
                  36.19986, 33.32033, 41.54757, 14.80903, 25.50445, 
                  48.12936, 23.859, 28.79535, 25.91581, 19.74538, 
                  44.01574, 28.79535, 38.66804, 26.32717, 37.43395])

# Province list:
Province = np.array(['P0', 'P1', 'P2', 'P3', 'P4', 
                      'P5', 'P6', 'P7', 'P8', 'P9', 
                      'P10', 'P11', 'P12', 'P13', 'P14', 
                      'P15', 'P16', 'P17', 'P18', 'P19'])

# Total seats:
s = 601

In [ ]:
#Remove the model print out 
env = gp.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()

prob_1_2=gp.Model("Question 2", env=env)
prob_1_2=gp.Model("Question 2")
#create variables to define matrix of variables
Qlen=len(Quota)
Plen=len(Province)
Poplen=len(Pop)

########## Create function variables ##########
#Allocation
a=prob_1_2.addVars(Qlen,vtype=GRB.INTEGER, name = ["Allocation "+i for i in Province])
#Max difference between each allocation and Quota
m=prob_1_2.addVars(Qlen,vtype=GRB.CONTINUOUS, name = ["difference "+i for i in Province])
#Max of m
z=prob_1_2.addVar(vtype=GRB.CONTINUOUS)

########## Set objective function ########## 

prob_1_2.setObjective(z,GRB.MINIMIZE)

########## Set Constraints ##########

# Setting m to |quota - allocation| 
for i in range(Qlen):  
    D=prob_1_2.addVar(lb=-GRB.INFINITY,ub=GRB.INFINITY)
    prob_1_2.addConstr(D==Quota[i]-a[i])
    prob_1_2.addGenConstrAbs(m[i],D,"abscon")

# z is equal to maximum of all differencece
prob_1_2.addConstr(z==max_(m[i] for i in range(Qlen)))
    
# All allocation must equal to the total quota
prob_1_2.addConstr(sum(a[i] for i in range(Qlen))==s)


########## Optimize the Problem ##########

#optimize problem
prob_1_2.optimize()

########## Print Results ##########
#print objective coeficients
#creeate dictionary to make dataframe
p1_2dict={}
alist=[]
mlist=[]
for i in a:
    alist.append(a[i].X)
    mlist.append(m[i].X)
p1_2dict["allocated"]=alist
p1_2dict["difference"]=mlist
df1_2=pd.DataFrame(p1_2dict,index=Province)
#Print data frame
df1_2

In [ ]:
# Print MinMax function optimal value
print("The optimized maximum difference between the quota and the allocation is {:,}".format(round(z.X,3)))

## <font color=#FF0000> Questions</font>
### (a) Explain why this linearization is valid. How many decision variables and how many con-straints does this mixed integer program have?

The linearization of this prolem is valid as instead of having ambiguous values for $a$ (the absolute value allows a to be positve or negative) we are replacing the ambiguity with a single decision variable $z$ that is equal to the maximum value of the abolute difference. $z$ will be the maximum difference regarless if $a$ is positive of negative therefore removing the non-linearity. We retain the information of the absolute difference between $q$ and $a$ by adding constraints. <br>

Since we only care about the positive difference between $q$ and $a$ the constraint linearization is valie. No matter the values of q<sub>i</sub> or a<sub>i</sub>, so long as q<sub>i</sub> != a<sub>i</sub> the resulting z<sub>i</sub> will always be larger some positive number. <br>
if q<sub>i</sub> > a<sub>i</sub>  then  z<sub>i</sub> > x , such that x>0 <br>
if q<sub>i</sub> < a<sub>i</sub>  then  z<sub>i</sub> < -x , such that -x>0 <br>
if q<sub>i</sub> = a<sub>i</sub>  then  z<sub>i</sub> = 0 <br><br>
The number of decsion variables has increased by 1 since b is a binary variablefor the big M method. Therefore in total there are **22 deciscion variables.**<br>
 20 variables for the allocation of representatives ( a<sub>i</sub> i=0,1..19 ) <br>
 20 variables for the difference between the abolute difference of allocation and quota ( m<sub>i</sub> i=0,1..19 ) <br>
 1 dummy variable to "linearize" the absolute difference between allocation and quota (D) <br>
 1 decision variable for the maximum value of the gap (z = max(m<sub>i</sub> i=0,1..19)) <br><br>
 
 **There are a total of 2 constraints:**<br>
 z<sub>i</sub> = |where q<sub>i</sub>-a<sub>i</sub>| i=0,1..19 <br>
 sum (a<sub>i</sub>) = s , where s is the total number of representatives<br>
 
    

### (b) (b) Find the maximum absolute gap between an allocation and quota in your optimal solution. Your code must be clear and easy to parse.
The maximum value of the abolute value of gaps is **0.504**

### 1.3: Political considerations.

In [1]:
# Population list:
import gurobipy as gp
from gurobipy import *
import numpy as np
import pandas as pd
Pop = np.array([5300000, 9200000, 5400000, 8000000, 3200000, 
                8800000, 8100000, 10100000, 3600000, 6200000, 
                11700000, 5800000, 7000000, 6300000, 4800000, 
                10700000, 7000000, 9400000, 6400000, 9100000])

# Quota (q_i) list:
Quota = np.array([21.80219, 37.84531, 22.21355, 32.90897, 13.16359, 
                  36.19986, 33.32033, 41.54757, 14.80903, 25.50445, 
                  48.12936, 23.859, 28.79535, 25.91581, 19.74538, 
                  44.01574, 28.79535, 38.66804, 26.32717, 37.43395])

# Province list:
Province = np.array(['P0', 'P1', 'P2', 'P3', 'P4', 
                      'P5', 'P6', 'P7', 'P8', 'P9', 
                      'P10', 'P11', 'P12', 'P13', 'P14', 
                      'P15', 'P16', 'P17', 'P18', 'P19'])

# Total seats:
s = 601

In [2]:
#suppress console output
env = gp.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()

#setup problem
prob_1_3=gp.Model("Question 3", env=env)
#prob_1_1=gp.Model("Question 1")

#create variables to define matrix of variables
Qlen=len(Quota)
Plen=len(Province)
Poplen=len(Pop)

########## Create function variables ##########
#Create variables for the the allocation
a=prob_1_3.addVars(Qlen,vtype=GRB.INTEGER, name = ["Allocation "+i for i in Province])
#create variable for the difference
z=prob_1_3.addVars(Qlen,vtype=GRB.CONTINUOUS, name = ["Abs Diff "+i for i in Province])
#create a binary variable b for the big M problem
b = prob_1_3.addVar(vtype=GRB.BINARY, name="b")

########## Set objective function ########## 
prob_1_3.setObjective(sum(z[i] for i in range(Qlen)),GRB.MINIMIZE)

########## Set Constraints ##########

## i P4 and P8 must have eqaul number of seats
prob_1_3.addConstr(a[4]==a[8])

## ii number of seats with provinces of more than 10M must not exeed quota
for i in [7,10,15]:
    prob_1_3.addConstr(a[i]<=Quota[i])
    
## iii number of seats allocated t P0 is more than quota then number allocated to P2 must be more than quota as well
## big M problem
M=100000000000000

#prob_1_3.addConstr(a[2]>Quota[2] if a[0]>Quota[0])
prob_1_3.addConstr(a[0] >= Quota[0] - M * (1 - b), name="bigM_constr1")
prob_1_3.addConstr(a[0] <= Quota[0] + M * b, name="bigM_constr2")

# Add indicator constraints
prob_1_3.addConstr((b == 1) >> ( a[2]>=Quota[2] ), name="province 2 bigger than quota")





## z is  |quota - allocation|
for i in range(Qlen):  
    D=prob_1_3.addVar(lb=-GRB.INFINITY,ub=GRB.INFINITY)
    prob_1_3.addConstr(D==Quota[i]-a[i])
    prob_1_3.addGenConstrAbs(z[i],D,"abscon")
    
## all allocation must be filled
prob_1_3.addConstr(sum(a[i] for i in range(Qlen))==s)



########## Optimize the Problem ##########
prob_1_3.optimize()

########## Print Results ##########
#print objective Coefficiencts
p1_3dict={}
alist=[]
zlist=[]
for i in a:
    alist.append(a[i].X)
    zlist.append(z[i].X)
p1_3dict["allocated"]=alist
p1_3dict["abs difference"]=zlist
df1_3=pd.DataFrame(p1_3dict,index=Province)
df1_3

,allocated,abs difference
P0,22.0,0.19781
P1,38.0,0.15469
P2,23.0,0.78645
P3,33.0,0.09103
P4,14.0,0.83641
P5,36.0,0.19986
P6,33.0,0.32033
P7,41.0,0.54757
P8,14.0,0.80903
P9,25.0,0.50445


In [3]:
print("The optimized absolute difference between the quota and the allocation is {:,}".format(round(prob_1_3.objVal,3)))

The optimized absolute difference between the quota and the allocation is 6.575


## <font color=#FF0000> Questions</font>
### a) Write down the mathematical formulas related to each consideration. How many decision variables and how many constraints does this mixed integer program have in total?
$i)$ <br>
    a<sub>4</sub>=a<sub>8</sub>  , where a<sub>i</sub> is the representative allocation for procinve i

$ii)$ <br>
    a<sub>7</sub><q<sub>7</sub> <br>
    a<sub>10</sub><q<sub>10</sub> <br>
    a<sub>15</sub><q<sub>15</sub> , where q<sub>i</sub> is the Quota for procinve i

$iii)$<br>
    a<sub>0</sub>>q<sub>0</sub> ->  a<sub>2</sub>>q<sub>2</sub>
   
The number of decsion variables has increased by 1 since b is a binary variablefor the big M method. Therefore in total there are **42 deciscion variables.**<br>
 20 variables for the allocation of representatives ( a<sub>i</sub> i=0,1..19 ) <br>
 20 variables for the difference between the abolute difference of allocation and quota ( z<sub>i</sub> i=0,1..19 ) <br>
 1 dummy variable to "linearize" the absolute difference between allocation and quota (D) <br>
 1 binary variable b for the big M method where b is 0 if a<sub>0</sub>>q<sub>0</sub> <br>
 Note: Big M is a fixed large value. <br><br>
 **There are a total of 7 constraints:**<br>
 z<sub>i</sub> = |where q<sub>i</sub>-a<sub>i</sub>| i=0,1..19 <br>
 sum (a<sub>i</sub>) = s , where s is the total number of representatives<br>
 a<sub>4</sub>=a<sub>8</sub> <br>
 a<sub>7</sub><q<sub>7</sub> <br>
 a<sub>10</sub><q<sub>10</sub> <br>
 a<sub>15</sub><q<sub>15</sub> <br>
 a<sub>0</sub>>q<sub>0</sub> ->  a<sub>2</sub>>q<sub>2</sub>
    

### (b) Find the minimum sum of absolute gaps using the optimal allocation. How much is the difference between your answer to this part and your answer to 1.1-(b)? Your code must be clear and easy to parse.

The minimum sum of the abolute gaps with the added constraints is 6.575. The difference with the previous optimization problem is **1.959** (6.575-4.616)

### 1.4: Opinions.

Politics is a complex field that deals directly with the voice of the people. It is crucial to preserve a sense of fairness or equity amount states. In a republic or represenative democracy like the United States the power of people and their voices is paramount. It seems clear that optimizing a system of representing people with constraints gives many possible solutions. Howeverm not all solutions are percieved as fair. It might even be correct to say that no optimal solution that eliminates the unhappiness of everyone exists <br><br>
Splitting the number of representatives by population means that goegraphies with larger populations have larger weights. Therefore larger states can over ride the voice of smaller states or even reduce the number of representatives to 0 if the population discrepancy is sufficent. I think that in order to counter balance the smaller voice of lower populated states each state is given the same number of senators  or voices. Therefore 1 state = 2 voices regardless of population. This system paired with the population distributed representatives balances the pitfals of both systems.

## 2: Did You Miss BalancedMilk Co. and Ms. Fraser?

### 2.1: The salesforce problem.

Mr. Lei draws on the following matrix (called **B**) to show among whom these effects exist and what their intensity is:


| | $S_1$ | $S_2$ | $S_3$ | $S_4$ | $S_5$ | $S_6$ | $S_7$ | $S_8$ | $S_9$ | $S_{X}$|
| -- | -- | -- | -- | -- | -- | -- | -- | -- | -- | -- |
|$S_1$|1 | 0.3 | -0.1 | 0 | 0  | -0.2  | 0  | -0.3 | 0.4 | 0.1 |
|$S_2$ | 0.2 | 1 | 0.1 | 0.1 | -0.1 | 0 | -0.1 | 0.1 | 0.3 | -0.2 |
|$S_3$| 0.2 | 0 | 1 | 0.1 | -0.1 | 0.2 | 0.2 | 0 | -0.3 | 0 |
|$S_4$| -0.1 | 0.2 | 0 | 1 | 0.2 | 0 | 0.2 | 0 | 0.1 | -0.1 |
|$S_5$| 0.1 | -0.1 | -0.1 | 0.1 | 1 | 0.1 | 0.1 | 0.3 | 0 | 0.1 | 
|$S_6$|-0.2| -0.2| -0.1| 0.2| 0.2| 1| 0.3| -0.1| 0| -0.2|
|$S_7$|0.1| 0.2| -0.1| -0.1| 0.2| -0.1| 1| 0.2| 0.1| 0.3|
|$S_8$|0.1| 0.1| 0.3| -0.1| 0.1| 0.2| 0.2| 1| 0.2| 0.1|
|$S_9$|0.3| 0.4| -0.2| 0.2| 0.1| 0| 0.1| -0.1| 1| -0.1|
|$S_{10}$|-0.1| -0.2| 0.1| 0.1| 0| -0.1| 0.2| 0.2| 0.3| 1|


where $a_{i,j}$ (the element in the intersection of the i'th row and the j'th column) is the payoff salesperson $i$ feels when salesperson $j$ receives one dollar. The payoff is linear in the received award. This means: $$\text{payoff}_i = \sum_{j = 1}^{10} a_{i,j} x_j$$


In [42]:
import gurobipy as gp
from gurobipy import *
import numpy as np
import pandas as pd

B = np.array([[1, 0.3, -0.1, 0, 0, -0.2, 0, -0.3, 0.4, 0.1],
              [0.2, 1, 0.1, 0.1, -0.1, 0, -0.1, 0.1, 0.3, -0.2],
              [0.2, 0, 1, 0.1, -0.1, 0.2, 0.2, 0, -0.3, 0],
              [-0.1, 0.2, 0, 1, 0.2, 0, 0.2, 0, 0.1, -0.1],
              [0.1, -0.1, -0.1, 0.1, 1, 0.1, 0.1, 0.3, 0, 0.1], 
              [-0.2, -0.2, -0.1, 0.2, 0.2, 1, 0.3, -0.1, 0, -0.2],
              [0.1, 0.2, -0.1, -0.1, 0.2, -0.1, 1, 0.2, 0.1, 0.3],
              [0.1, 0.1, 0.3, -0.1, 0.1, 0.2, 0.2, 1, 0.2, 0.1],
              [0.3, 0.4, -0.2, 0.2, 0.1, 0, 0.1, -0.1, 1, -0.1],
              [-0.1, -0.2, 0.1, 0.1, 0, -0.1, 0.2, 0.2, 0.3, 1]])

names=['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8','S_9','S_10']



num_people=len(B)

prob_2_1=gp.Model("Question 2.1")

########## Create function variables ##########

X2_1=prob_2_1.addVars(num_people,lb=0,vtype=GRB.CONTINUOUS,name = [i for i in names])
z=prob_2_1.addVar(lb=0,vtype=GRB.CONTINUOUS,name = "dummy var")
bmin = prob_2_1.addVars(num_people,vtype=GRB.BINARY, name=["bmin" +str(i) for i in range(num_people)])
bmax = prob_2_1.addVars(num_people,vtype=GRB.BINARY, name=["bmax" +str(i) for i in range(num_people)])


########## Set objective function ########## 

#prob_2_1.setObjective(z,GRB.MAXIMIZE)
prob_2_1.setObjective(sum(B[i][j]*X2_1[j] for i in range(num_people) for j in range(num_people))/num_people,GRB.MAXIMIZE)


########## Set Constraints ##########

prob_2_1.addConstr(sum(X2_1[j] for j in range(num_people))<=10000)

for i in range(num_people):
    prob_2_1.addConstr(sum(X2_1[j]*B[i][j] for j in range(num_people))>= 1500-500*bmin[i])
    prob_2_1.addConstr(sum(X2_1[j]*B[i][j] for j in range(num_people))>= 2500*bmax[i])
#at most 3 people can recieve less than 1000 units
prob_2_1.addConstr(sum(bmin[i] for i in range(num_people))<=3)
# 3 people must recieve more than 2500 units
prob_2_1.addConstr(sum(bmax[i] for i in range(num_people))==3)

    
########## Optimize the Problem ##########

prob_2_1.optimize()


########## Print Results ##########

for i in prob_2_1.getVars():
    if i.x>0 and "S" in i.varName:
        print(i.varName,i.x)
prob_2_1.printAttr(['X', 'Obj'])

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 23 rows, 31 columns and 222 nonzeros
Model fingerprint: 0x420f1341
Variable types: 11 continuous, 20 integer (20 binary)
Coefficient statistics:
  Matrix range     [1e-01, 3e+03]
  Objective range  [9e-02, 2e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 1e+04]
Presolve removed 0 rows and 1 columns
Presolve time: 0.00s
Presolved: 23 rows, 30 columns, 222 nonzeros
Variable types: 10 continuous, 20 integer (20 binary)

Root relaxation: objective 2.052700e+03, 23 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 2052.70000    0    6          - 2052.70000      -     -    0s
H    0     0                    1712.1480153 2052.70000  19.9%     -    0s
H    0     0      

In [39]:
print("The optimized maximum average units is therefore {:,} per person".format(round(prob_2_1.objVal,2)))

The optimized maximum average units is therefore 1,950.1 per person


### 2.2: BalancedMilk in the middle of the crisis.

In [ ]:
#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd
import numpy as np
#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()



Cost_2_2 = np.array([[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
                [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
                [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
                [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
                [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
                [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
                [12, 19,  5, 28, 47, 39, 15, 35, 9, 51],
                [34, 17, 10, 21, 9, 33, 14, 26, 19, 45]])

#since supply center 8 is not producing dummify and change price to 15 forcing non-production
Cost_2_2[7]=[15, 15, 15, 15, 15, 15, 15, 15, 15, 15]

#modify the amount to supply to fit the constraints more easily
#440 added within the penalty
SupplyCenter_2_2 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8']
Supply_2_2 = [110, 80, 100, 240, 100, 280, 130, 0]

DemandMarket_2_2 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand_2_2 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

# Your solution for 2.2

#setup problem
prob_2_2=gb.Model("Question 2.2",env=env)
Slen_2_2=len(Supply_2_2)
Dlen_2_2=len(Demand_2_2)
penalty_2_2=15

########## Create function variables ##########
#Create variables for each demand and supply
X_2_2=prob_2_2.addVars(Slen_2_2,Dlen_2_2,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter_2_2 for j in DemandMarket_2_2])
#create a binary variable b for the big M problem
b = prob_2_2.addVars(Slen_2_2,vtype=GRB.BINARY, name=["b" +str(i) for i in range(Slen_2_2)])

########## Set objective function ########## 
#add the cost of delivery
#the cost of not dlievering (the penalty)
#the cost of S8 being non-operational
prob_2_2.setObjective(sum(Cost_2_2[i][j]*X_2_2[i,j] for i in range(Slen_2_2)for j in range (Dlen_2_2)) + 
                      (penalty_2_2*(sum(Supply_2_2)-sum(X_2_2[i,j] for i in range(Slen_2_2)for j in range (Dlen_2_2))))+
                      15*440,GRB.MINIMIZE)


########## Set Constraints ##########

## Supplied amount must not exceed total amount for supply
for i in range(Slen_2_2):
    prob_2_2.addConstr(sum(X_2_2[i,j] for j in range (Dlen_2_2))<=Supply_2_2[i])
## Demand cannot be overfilled for each demand centre
for j in range(Dlen_2_2):
    prob_2_2.addConstr(sum(X_2_2[i,j] for i in range (Slen_2_2))<=Demand_2_2[j])
    
## Constraint 3 S3, S4 & S5 two need to have above 50% utilization
#Note: indexing in python starts at 0 therefore i+1= demand or supplier's number
## big M problem
M=100000000000000
l=[2,3,4]

#sum of binary variables need to be at least 2 therefore 2 of the 3 will be required
prob_2_2.addConstr(sum(b[i] for i in range(3))>=2)
for i in range(3):
    #set k to the proper supply center. Numbers are offset by 1 due to python indexing
    k=l[i]
    prob_2_2.addConstr(sum(X_2_2[k,j] for j in range(Slen_2_2))>=.5*Supply_2_2[k]-M*b[i])


## Constraint 4 D5>80% , D6>60% or both
## big M problem


#prob_1_3.addConstr(a[2]>Quota[2] if a[0]>Quota[0])
prob_2_2.addConstr(sum(X_2_2[i,4] for i in range(Slen_2_2)) >=0.8*Demand_2_2[4] - M * (1 - b[4]), name="bigM_constr1")
prob_2_2.addConstr(sum(X_2_2[i,5] for i in range(Slen_2_2)) >=0.6*Demand_2_2[5] + M * b[4], name="bigM_constr2")


## Constraint 5 If D2 >50% then D8,D9 >50% 
#validated
# temporary variables
u = prob_2_2.addVar(lb=-GRB.INFINITY, name='u')
z = prob_2_2.addVar(vtype=GRB.BINARY, name='z')

# Set u equal to sum of demand
prob_2_2.addConstr(u == sum(X_2_2[i,1] for i in range(Slen_2_2)))

#if D2 less than 50% then z=0 therefore D8 & D9 do not need to be above 50% of their demand
#Note: indexing in python starts at 0 therefore i+1= demand or supplier's number
prob_2_2.addConstr((z == 0) >> (u >= (0.49999*Demand_2_2[1])))
prob_2_2.addConstr((z == 1) >> (u <= (0.5*Demand_2_2[1])))
prob_2_2.addConstr((z == 0) >> ((sum(X_2_2[i,7] for i in range(Slen_2_2)) >= 0.5*Demand_2_2[7])))
prob_2_2.addConstr((z == 0) >> ((sum(X_2_2[i,8] for i in range(Slen_2_2)) >= 0.5*Demand_2_2[8])))

########## Optimize the Problem ##########

prob_2_2.optimize()


########## Print Results ##########

pb_2_2dict={}
l_2_2=[]
for i in X_2_2:
    col=("D_"+str(i[1]+1))      
    if col in pb_2_2dict.keys():
        pb_2_2dict[col].append(round(X_2_2[i].X))
    else:
        l_2_2=[]
        pb_2_2dict[col]=l_2_2
        pb_2_2dict[col].append(round(X_2_2[i].X))

df_2_2=pd.DataFrame(pb_2_2dict,index=SupplyCenter_2_2)
print("\n")
print("The optimized minmum total cost is therefore ${:,}".format(round(prob_2_2.objVal)))
print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df_2_2)

In [ ]:
for i in prob_2_2.getVars():
    if i.x>0:
        print(i.varName,i.x)

## <font color=#FF0000> Questions</font>
### a) What is the lowest cost, the utilized capacity of each supply center, and the satisfied demand of each demand center?

The lowest cost overall cost is \\$15,918. The optimized distribution is listed in the table above.

### 2.3 - BalancedMilk before the next crisis.

In [6]:
#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd
import numpy as np

Cost_2_3 = np.array([[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
                [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
                [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
                [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
                [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
                [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
                [12, 19,  5, 28, 47, 39, 15, 35, 9, 51],
                [34, 17, 10, 21, 9, 33, 14, 26, 19, 45]])

SupplyCenter_2_3 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8']
Supply_2_3 = [110, 80, 100, 240, 100, 280, 130, 440]
DemandMarket_2_3 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand_2_3 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

# Your solution for 2.3




In [41]:
#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()


#setup problem
prob_2_3_min=gb.Model("Question 2.3_min" ,env=env)
prob_2_3_max=gb.Model("Question 2.3_max" ,env=env)

Slen_2_3=len(Supply_2_3)
Dlen_2_3=len(Demand_2_3)
penalty_2_3=30
plist=[prob_2_3_min]

########## Create function variables ##########
X_2_3_min = prob_2_3_min.addVars(Slen_2_3,Dlen_2_3,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter_2_3 for j in DemandMarket_2_3])
    #create a binary variable b for the big M problems
b_2_3_min = prob_2_3_min.addVars(Slen_2_3,vtype=GRB.BINARY, name=["b" +str(i) for i in range(Slen_2_3)])

########## Set objective function ########## 
prob_2_3_min.setObjective(sum(b_2_3_min[i]*Cost_2_3[i][j]*X_2_3_min[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3)) + 
                    (penalty_2_3*(sum(Supply_2_3)-sum(X_2_3_min[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3))))
                        ,GRB.MINIMIZE)

#MAXIMIZE
########## Set Constraints ##########

#set constraints
##setting the supply constrains for each supply centre
for i in range(Slen_2_3):
    prob_2_3_min.addConstr(sum(X_2_3_min[i,j] for j in range (Dlen_2_3))==b_2_3_min[i]*Supply_2_3[i])
    
##setting the demand constrains for each demand centre with no over supply of demand
for j in range(Dlen_2_3):
    prob_2_3_min.addConstr(sum(X_2_3_min[i,j] for i in range (Slen_2_3))<=Demand_2_3[j])
    
#setting binary variables

prob_2_3_min.addConstr(sum(b_2_3_min[i] for i in range(Slen_2_3))<=Slen_2_3-1)

########## Optimize the Problem ##########
prob_2_3_min.optimize()

########## Print Results ##########
def Print_2_3(X,SupplyCenter=SupplyCenter_2_3):
    pb_dict={}
    l=[]
    for i in X:
        col=("D_"+str(i[1]+1))      
        if col in pb_dict.keys():
            pb_dict[col].append(round(X[i].X))
        else:
            l=[]
            pb_dict[col]=l
            pb_dict[col].append(round(X[i].X))

    df=pd.DataFrame(pb_dict,index=SupplyCenter)
    return(df)



print("Shutting down 1 Centre:")
print(Print_2_3(X_2_3_min))



Shutting down 1 Centre:
     D_1  D_2  D_3  D_4  D_5  D_6  D_7  D_8  D_9  D_10
S_1    0    0    0    0    0    0    0    0  110     0
S_2    0    0    0    0    0    0    0    0    0     0
S_3    0    0    0    0    0  100    0    0    0     0
S_4    0  100    0    0    0  140    0    0    0     0
S_5   90    0    0   10    0    0    0    0    0     0
S_6    0    0    0    0  180    0    0   30    0    70
S_7    0    0   80    0    0    0    0    0   50     0
S_8    0    0   70  160    0    0  210    0    0     0


In [42]:

for m in range(8):
    for n in range(8):
        for o in range(8):
            if n!=m and n!=o and m!=o:
                #print(m,n,o)
                #suppress console output
                env = gb.Env(empty=True)
                env.setParam("OutputFlag",0)
                env.start()


                #setup problem
                prob_2_3_max=gb.Model("Question 2.3_max" ,env=env)

                Slen_2_3=len(Supply_2_3)
                Dlen_2_3=len(Demand_2_3)
                penalty_2_3=30
               

                ########## Create function variables ##########
                X_2_3_max = prob_2_3_max.addVars(Slen_2_3,Dlen_2_3,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter_2_3 for j in DemandMarket_2_3])
                    #create a binary variable b for the big M problems
                b_2_3_max = prob_2_3_max.addVars(Slen_2_3,vtype=GRB.BINARY, name=["b" +str(i) for i in range(Slen_2_3)])

                ########## Set objective function ########## 
                prob_2_3_max.setObjective(sum(b_2_3_max[i]*Cost_2_3[i][j]*X_2_3_max[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3)) + 
                                    (penalty_2_3*(sum(Supply_2_3)-sum(X_2_3_max[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3))))
                                        ,GRB.MINIMIZE)
                #MAXIMIZE
                ########## Set Constraints ##########

                #set constraints
                ##setting the supply constrains for each supply centre
                for i in range(Slen_2_3):
                    prob_2_3_max.addConstr(sum(X_2_3_max[i,j] for j in range (Dlen_2_3))==b_2_3_max[i]*Supply_2_3[i])

                ##setting the demand constrains for each demand centre with no over supply of demand
                for j in range(Dlen_2_3):
                    prob_2_3_max.addConstr(sum(X_2_3_max[i,j] for i in range (Slen_2_3))<=Demand_2_3[j])

                #setting binary variables

                prob_2_3_min.addConstr(sum(b_2_3_min[i] for i in range(Slen_2_3))>=Slen_2_3-3)
                prob_2_3_max.addConstr((b_2_3_max[m]==0))
                prob_2_3_max.addConstr((b_2_3_max[n]==0))
                prob_2_3_max.addConstr((b_2_3_max[o]==0))

                ########## Optimize the Problem ##########
                prob_2_3_max.optimize()
                
                if round(prob_2_3_max.objVal)>minval:
                    minval=prob_2_3_max.objVal
                    mmax=m
                    nmax=n
                    omax=o

#run model with new variables
prob_2_3_max=gb.Model("Question 2.3_max" ,env=env)

Slen_2_3=len(Supply_2_3)
Dlen_2_3=len(Demand_2_3)
penalty_2_3=30
               

                ########## Create function variables ##########
X_2_3_max = prob_2_3_max.addVars(Slen_2_3,Dlen_2_3,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter_2_3 for j in DemandMarket_2_3])
                    #create a binary variable b for the big M problems
b_2_3_max = prob_2_3_max.addVars(Slen_2_3,vtype=GRB.BINARY, name=["b" +str(i) for i in range(Slen_2_3)])

                ########## Set objective function ########## 
prob_2_3_max.setObjective(sum(b_2_3_max[i]*Cost_2_3[i][j]*X_2_3_max[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3)) + 
                          (penalty_2_3*(sum(Supply_2_3)-sum(X_2_3_max[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3))))
                          ,GRB.MINIMIZE)
                #MAXIMIZE
                ########## Set Constraints ##########

                #set constraints
                ##setting the supply constrains for each supply centre
for i in range(Slen_2_3):
    prob_2_3_max.addConstr(sum(X_2_3_max[i,j] for j in range (Dlen_2_3))==b_2_3_max[i]*Supply_2_3[i])

                ##setting the demand constrains for each demand centre with no over supply of demand
for j in range(Dlen_2_3):
    prob_2_3_max.addConstr(sum(X_2_3_max[i,j] for i in range (Slen_2_3))<=Demand_2_3[j])

                #setting binary variables

prob_2_3_min.addConstr(sum(b_2_3_min[i] for i in range(Slen_2_3))>=Slen_2_3-3)
prob_2_3_max.addConstr((b_2_3_max[mmax]==0))
prob_2_3_max.addConstr((b_2_3_max[nmax]==0))
prob_2_3_max.addConstr((b_2_3_max[omax]==0))

########## Optimize the Problem ##########
prob_2_3_max.optimize()                   
print(prob_2_3_max.objVal)     
print("\n")
print("Shutting down 3 Centres:")
print(Print_2_3(X_2_3_max))

32680.0


Shutting down 3 Centres:
     D_1  D_2  D_3  D_4  D_5  D_6  D_7  D_8  D_9  D_10
S_1    0    0    0    0  110    0    0    0    0     0
S_2    0    0   20    0    0   60    0    0    0     0
S_3    0    0    0    0    0   30    0    0    0    70
S_4    0    0    0    0    0    0    0    0    0     0
S_5   90    0    0    0    0    0    0    0   10     0
S_6    0    0    0    0    0    0    0    0    0     0
S_7    0    0  130    0    0    0    0    0    0     0
S_8    0    0    0    0    0    0    0    0    0     0


In [44]:
for j in plist:
    print(j)
    print("The optimized minmum total cost is therefore ${:,}".format(round(j.objVal)))
    print("The following binary varibales indicate which supply centers should be shut down")
    for i in j.getVars():
        if "b" in i.varName and i.X==0:
            print(i.varName,i.x)
    print("The optimized minmum total cost is therefore ${:,}".format(round(prob_2_3_max.objVal)))
    print("The following binary varibales indicate which supply centers should be shut down")
    for i in prob_2_3_max.getVars():
        if "b" in i.varName and i.X==0:
            print(i.varName,i.x)
print("\n","\n")            
print("Note: becasue of python indexing b0 corresponds to S1, b1 to S2 ect")   

<gurobi.Model MIP instance Question 2.3_min: 19 constrs, 88 vars, Parameter changes: OutputFlag=0>
The optimized minmum total cost is therefore $19,460
The following binary varibales indicate which supply centers should be shut down
b1 0.0
The optimized minmum total cost is therefore $32,680
The following binary varibales indicate which supply centers should be shut down
b3 0.0
b5 0.0
b7 0.0

 

Note: becasue of python indexing b0 corresponds to S1, b1 to S2 ect


## <font color=#FF0000> Questions</font>
### Which supply center is shut down in the best case? Which three supply centers are shut down in the worst case? What is the cost in each case? 

In the best case scenario S2 should be shut down to minimize the overall cost. The minimized cost with these constraints is \\$19,460

The the supply centers that should be shut down in he worst case scenario are S4,S6 and S8. The minimized cost for the worst case scenario is \\$23,020

The optimized minimim costs are:<br>
**Best Case: \\$19,460**

**Worst Case: \\$32,680.0**

In [20]:
slist=[]
minval=0
Slen_2_3=4

In [29]:

for m in range(8):
    for n in range(8):
        for o in range(8):
            if n!=m and n!=o and m!=o:
                #print(m,n,o)
                #suppress console output
                env = gb.Env(empty=True)
                env.setParam("OutputFlag",0)
                env.start()


                #setup problem
                prob_2_3_max=gb.Model("Question 2.3_max" ,env=env)

                Slen_2_3=len(Supply_2_3)
                Dlen_2_3=len(Demand_2_3)
                penalty_2_3=30
               

                ########## Create function variables ##########
                X_2_3_max = prob_2_3_max.addVars(Slen_2_3,Dlen_2_3,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter_2_3 for j in DemandMarket_2_3])
                    #create a binary variable b for the big M problems
                b_2_3_max = prob_2_3_max.addVars(Slen_2_3,vtype=GRB.BINARY, name=["b" +str(i) for i in range(Slen_2_3)])

                ########## Set objective function ########## 
                prob_2_3_max.setObjective(sum(b_2_3_max[i]*Cost_2_3[i][j]*X_2_3_max[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3)) + 
                                    (penalty_2_3*(sum(Supply_2_3)-sum(X_2_3_max[i,j] for i in range(Slen_2_3)for j in range (Dlen_2_3))))
                                        ,GRB.MINIMIZE)
                #MAXIMIZE
                ########## Set Constraints ##########

                #set constraints
                ##setting the supply constrains for each supply centre
                for i in range(Slen_2_3):
                    prob_2_3_max.addConstr(sum(X_2_3_max[i,j] for j in range (Dlen_2_3))==b_2_3_max[i]*Supply_2_3[i])

                ##setting the demand constrains for each demand centre with no over supply of demand
                for j in range(Dlen_2_3):
                    prob_2_3_max.addConstr(sum(X_2_3_max[i,j] for i in range (Slen_2_3))<=Demand_2_3[j])

                #setting binary variables

                prob_2_3_min.addConstr(sum(b_2_3_min[i] for i in range(Slen_2_3))>=Slen_2_3-3)
                prob_2_3_max.addConstr((b_2_3_max[m]==0))
                prob_2_3_max.addConstr((b_2_3_max[n]==0))
                prob_2_3_max.addConstr((b_2_3_max[o]==0))

                ########## Optimize the Problem ##########
                prob_2_3_max.optimize()
                
                if round(prob_2_3_max.objVal)>minval:
                    minval=prob_2_3_max.objVal
                    mmax=m
                    nmax=n
                    omax=o

In [30]:
print(mmax)
print(nmax)
print(omax)
print(minval)

3
5
7
32680.0
